# Risk Parity - Long Only Equal Weight Strategy

This notebook demonstrates a long-only risk parity strategy using A-Stock ETFs with pysystemtrade.

**Instruments:**
- 510300.SH: Huatai-PB CSI 300 ETF (Equity)
- 518880.SH: Gold ETF (Commodity)
- 511260.SH: SSE Corporate Bond ETF (Fixed Income)

**Risk parity achieved through:**
1. Trading rule: constant 10.0 forecast (long bias)
2. Equal weights: pysystemtrade uses 1/n when no instrument_weights configured
3. Vol scaling: positionSize stage divides notional by instrument volatility

In [ ]:
# Change to project root directory
import os
from pathlib import Path

# Navigate from examples/ to project root
project_root = Path(os.getcwd()).parent
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")



In [ ]:
# Setup: ensure target ETFs are in the universe

TARGET_ETFS = ["510300.SH", "518880.SH", "511260.SH"]

import pandas as pd
from pathlib import Path

# Check valid_symbols.csv
valid_symbols_path = Path("data/astock/csvconfig/valid_symbols.csv")
if valid_symbols_path.exists():
    valid_symbols = pd.read_csv(valid_symbols_path)
    if "Instrument" in valid_symbols.columns:
        existing = valid_symbols["Instrument"].tolist()
        missing = [e for e in TARGET_ETFS if e not in existing]
        if missing:
            print(f"Adding to valid_symbols.csv: {missing}")
            new_rows = pd.DataFrame({"Instrument": missing})
            valid_symbols = pd.concat([valid_symbols, new_rows], ignore_index=True)
            valid_symbols.to_csv(valid_symbols_path, index=False)
        else:
            print("All target ETFs already in valid_symbols.csv")
    else:
        print("Invalid format, recreating...")
        valid_symbols = pd.DataFrame({"Instrument": TARGET_ETFS})
        valid_symbols.to_csv(valid_symbols_path, index=False)
else:
    print("valid_symbols.csv not found, creating...")
    valid_symbols = pd.DataFrame({"Instrument": TARGET_ETFS})
    valid_symbols.to_csv(valid_symbols_path, index=False)

print(f"Target ETFs: {TARGET_ETFS}")

In [ ]:
# Fetch data for target ETFs

import subprocess

# Check if parquet data exists for target ETFs
parquet_dir = Path("data/astock/daily_prices_parquet")
existing_files = list(parquet_dir.glob("*.parquet")) if parquet_dir.exists() else []

print(f"Existing parquet files: {len(existing_files)}")

# Fetch data using the fetcher (using --symbols for specific instruments)
print("
Fetching data for target ETFs...")
print("This may take a few minutes...")

result = subprocess.run(
    ["python", "-m", "sysinit.astock.fetcher",
     "--symbols"] + TARGET_ETFS +
    ["--once", "--freq", "daily"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Data fetch completed!")
    if result.stdout:
        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
else:
    print(f"Error fetching data: {result.stderr}")
    print("
You may need to set XIXIMIAO_TOKEN in your .env file")

In [ ]:
# Verify data is available

from sysdata.sim.astock_sim_data import AStockSimData

data = AStockSimData()
available = data.get_instrument_list()

print(f"Available instruments: {len(available)}")

for etf in TARGET_ETFS:
    if etf in available:
        prices = data.get_raw_price(etf)
        print(f"  {etf}: {len(prices)} rows, {prices.index[0].date()} to {prices.index[-1].date()}")
    else:
        print(f"  {etf}: NOT AVAILABLE")

In [ ]:
# Create and run the risk parity system

from systems.basesystem import System
from systems.rawdata import RawData
from systems.trading_rules import TradingRule
from systems.forecasting import Rules
from systems.provided.rules.long_only import long_only
from sysdata.config.configdata import Config

# Create config with target instruments
config = Config()
config.instruments = TARGET_ETFS
config.notional_trading_capital = 1000000  # 1M CNY

# Create rules with long_only trading rule
rules = Rules({"long_only": TradingRule(long_only)})

# Create system with Rules stage
system = System([RawData(), rules], data=data, config=config)

print("System created successfully!")
print(f"Instruments in system: {system.get_instrument_list()}")

In [ ]:
# Display forecasts

print("=" * 60)
print("Risk Parity Strategy - Long Only Equal Weight")
print("=" * 60)

print("\nForecast (should be constant 10.0):")
for instr in TARGET_ETFS:
    if instr in system.get_instrument_list():
        forecast = system.rules.get_raw_forecast(instr, "long_only")
        print(f"  {instr}: {forecast.iloc[-1]:.4f}")
    else:
        print(f"  {instr}: not in system")

In [ ]:
# Display equal weights

print("\nEqual weights (1/n each):")
weights = system.portfolio.get_instrument_weights()
print(weights[TARGET_ETFS].tail())

In [ ]:
# Display vol-scaled positions

print("\nPosition sizes (vol-scaled):")
for instr in TARGET_ETFS:
    if instr in system.get_instrument_list():
        pos = system.positionSize.get_actual_position(instr)
        vol = system.positionSize.get_instrument_currency_vol(instr).iloc[-1]
        print(f"  {instr}: position={pos.iloc[-1]:.4f}, vol={vol:.6f}")

In [ ]:
# Run backtest with full system

from systems.forecast_scale_cap import ForecastScaleCap
from systems.forecast_combine import ForecastCombine
from systems.positionsizing import PositionSizing
from systems.portfolio import Portfolios
from systems.accounts.accounts_stage import Account

# Create full system with all stages for backtesting
stages = [
    Account(),              # Account stage for P&L
    ForecastScaleCap(),    # Scale forecasts
    Rules({"long_only": TradingRule(long_only)}),  # Trading rule
    ForecastCombine(),     # Combine forecasts (not used with single rule)
    PositionSizing(),     # Position sizing
    Portfolios(),          # Portfolio weights
    RawData(),            # Raw data
]

backtest_system = System(stages, data=data, config=config)

print("Backtest system created!")
print(f"Capital: {config.notional_trading_capital:,.0f} CNY")

In [ ]:
# Calculate and display performance metrics

import numpy as np

print("=" * 60)
print("BACKTEST PERFORMANCE METRICS")
print("=" * 60)

# Get equity curve
equity_curve = backtest_system.accounts.portfolio().curve()

# Calculate returns
returns = equity_curve.pct_change().dropna()

# Annualized metrics (assuming 252 trading days)
trading_days_per_year = 252

# Total return
total_return = (equity_curve.iloc[-1] / equity_curve.iloc[0] - 1) * 100

# Annualized return
years = len(equity_curve) / trading_days_per_year
annualized_return = ((1 + total_return/100) ** (1/years) - 1) * 100 if years > 0 else 0

# Volatility (annualized)
annualized_vol = returns.std() * np.sqrt(trading_days_per_year) * 100

# Sharpe Ratio (assuming 0% risk-free rate)
sharpe_ratio = annualized_return / annualized_vol if annualized_vol > 0 else 0

# Drawdown
cumulative = (1 + returns).cumprod()
running_max = cumulative.cummax()
drawdown = (cumulative - running_max) / running_max * 100
max_drawdown = drawdown.min()

# Sortino Ratio (downside deviation)
negative_returns = returns[returns < 0]
downside_dev = negative_returns.std() * np.sqrt(trading_days_per_year) * 100 if len(negative_returns) > 0 else 0
sortino_ratio = annualized_return / downside_dev if downside_dev > 0 else 0

# Calmar Ratio (return / max drawdown)
calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0

print(f"\nPeriod: {equity_curve.index[0].date()} to {equity_curve.index[-1].date()}")
print(f"Trading days: {len(equity_curve)}")
print(f"Years: {years:.2f}")
print(f"\n--- Returns ---")
print(f"Total Return: {total_return:.2f}%")
print(f"Annualized Return: {annualized_return:.2f}%")
print(f"Annualized Volatility: {annualized_vol:.2f}%")
print(f"\n--- Risk Metrics ---")
print(f"Max Drawdown: {max_drawdown:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")

In [ ]:
# Plot equity curve and drawdown

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Equity curve
axes[0].plot(equity_curve.index, equity_curve.values, 'b-', linewidth=1.5)
axes[0].set_ylabel('Portfolio Value (CNY)')
axes[0].set_title('Risk Parity Strategy - Long Only Equal Weight')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=config.notional_trading_capital, color='gray', linestyle='--', alpha=0.5, label='Initial Capital')
axes[0].legend()

# Drawdown
axes[1].fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.3)
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Date')
axes[1].set_title('Drawdown')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Display per-instrument contribution

print("\n--- Per-Instrument Performance ---")

for instr in TARGET_ETFS:
    if instr in backtest_system.get_instrument_list():
        # Get instrument account curve
        instr_curve = backtest_system.accounts.instrument(instr).curve()
        
        if len(instr_curve) > 0:
            instr_return = (instr_curve.iloc[-1] / instr_curve.iloc[0] - 1) * 100
            instr_vol = instr_curve.pct_change().dropna().std() * np.sqrt(trading_days_per_year) * 100
            instr_sharpe = instr_return / instr_vol if instr_vol > 0 else 0
            
            print(f"\n{instr}:")
            print(f"  Total Return: {instr_return:.2f}%")
            print(f"  Annualized Vol: {instr_vol:.2f}%")
            print(f"  Sharpe: {instr_sharpe:.2f}")